# Chicken Egg Quality Analysis - YOLO Classification Training
This notebook is configured to run on Kaggle with Dual T4 GPUs.
It trains a YOLO26 nano **classification** model on the chicken egg dataset. We first reorganize the Kaggle dataset into the folder structure required by YOLO classification, then train the model and export it to ONNX.

In [ ]:
!pip install ultralytics onnx onnxruntime albumentations
# Albumentations is installed to support advanced augmentations

In [ ]:
import os
import yaml
from ultralytics import YOLO
from IPython.display import Image, display

# Verify GPU availability
!nvidia-smi

## 1. Dataset Reorganization
The provided Kaggle dataset places all images directly into `training/` and `testing/` directories without class subfolders, and uses `info.labels` to map images to their classes. Ultralytics YOLO Classification requires images to be separated into class subfolders (e.g., `dataset/train/fertile/img.jpg`).

The code below reads `info.labels` and reorganizes the images into the proper structure inside `/kaggle/working/dataset`.

In [ ]:
import json
import shutil
from pathlib import Path

source_dataset_dir = '/kaggle/input/datasets/hlnkmb/chicken-egg-analysis-dataset'
labels_file = os.path.join(source_dataset_dir, 'info.labels')
output_dataset_dir = '/kaggle/working/dataset'

with open(labels_file, 'r') as f:
    info = json.load(f)

# Create class folders for train and val splits
for split in ['train', 'val']:
    for cls in ['fertile', 'infertile', 'dead']:
        os.makedirs(os.path.join(output_dataset_dir, split, cls), exist_ok=True)

copied_count = 0
for file_info in info['files']:
    src_path = os.path.join(source_dataset_dir, file_info['path'])
    label = file_info['label']['label']
    category = file_info['category']
    
    # Map 'training' to 'train' and 'testing' to 'val'
    split = 'train' if category == 'training' else 'val'
    
    dst_path = os.path.join(output_dataset_dir, split, label, os.path.basename(src_path))
    
    if os.path.exists(src_path):
        shutil.copy(src_path, dst_path)
        copied_count += 1

print(f"Successfully organized {copied_count} images into {output_dataset_dir}")

## 2. Model Training
Now that the dataset is organized, we can train the YOLO classification model.
Note: We use `yolo26n-cls.pt` (the classification variant).

In [ ]:
# Initialize YOLO model (using YOLO26 nano classification base)
model = YOLO('yolo26n-cls.pt') 

# Train the model
results = model.train(
    data='/kaggle/working/dataset', # Path to the reorganized dataset directory
    epochs=50, # Adjust based on convergence
    imgsz=224, # Standard size for classification
    batch=32,  # Increased batch size for Dual T4 GPUs
    device=[0, 1], # Uses both GPUs
    
    # Classification Augmentations
    degrees=15.0,        # rotation
    scale=0.5,           # scaling
    fliplr=0.5,          # flip horizontal
    hsv_h=0.015,         # color transformations (hue)
    hsv_s=0.7,           # color transformations (saturation)
    hsv_v=0.4,           # color transformations (value)
    
    # Other params
    project='/kaggle/working/runs',
    name='egg_quality_cls_model'
)

## 3. Evaluation
Ultralytics automatically generates evaluation plots. For classification, these include accuracy curves and a confusion matrix.

In [ ]:
run_dir = results.save_dir

plots = ['confusion_matrix.png', 'results.png']

for plot in plots:
    plot_path = os.path.join(run_dir, plot)
    if os.path.exists(plot_path):
        print(f"Displaying {plot}:")
        display(Image(filename=plot_path, width=800))
    else:
        print(f"{plot} not found.")

## 4. Export to ONNX with INT8 Quantization
Convert the model to ONNX format with INT8 quantization.

In [ ]:
# Export the model
exported_path = model.export(
    format='onnx', 
    int8=True,       # INT8 Quantization
    simplify=True,   # Simplifies the ONNX model
    imgsz=224
)

print(f"Model successfully exported for edge deployment to: {exported_path}")